In [3]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [111]:
import pandas as pd
from datetime import datetime
from collections import defaultdict

# Define time period
start = "2025-04-01T00:00:00Z"  # adjust as needed

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org', 'HTOC Comm', 'HTOC legacy data']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=threatAssess,associatedGroups,tags,observations&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")

observed_src


Querying owner: HTOC Org

Querying owner: HTOC Comm

Querying owner: HTOC legacy data

Retrieved 8556 unique observed indicators.


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,whoisActive,sha256,md5,sha1,size,url,Subject,text,address,indicator
0,5629499536010176,2025-03-14T11:45:58Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-06T11:36:05Z,4.0,69.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.142.193.247
1,4553658,2024-03-29T13:28:05Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-06T11:36:05Z,3.0,6.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,149.88.17.136
2,4553619,2024-03-29T13:13:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-06T11:36:03Z,3.0,7.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,169.150.196.153
3,4501750,2024-01-18T14:44:39Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-06T11:35:51Z,3.0,50.0,1.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.42.116.192
4,6755399442004904,2025-02-12T19:49:28Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-06T11:35:45Z,3.0,76.0,1.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,118.193.59.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,4524882,2024-02-21T16:49:47Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-02T07:35:29Z,3.0,13.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.91.126.46
9996,4524861,2024-02-21T16:49:47Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-02T07:35:29Z,3.0,13.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.179.189.45
9997,4524855,2024-02-21T16:49:47Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-02T07:35:29Z,3.0,13.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.164.172.220
9998,4524029,2024-02-20T14:59:53Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-02T07:35:29Z,3.0,9.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39.144.144.11


In [208]:
import pandas as pd
from datetime import datetime, timedelta

# Initialize an empty DataFrame to store filtered tags
filtered_tags = pd.DataFrame()

# Loop through each row in observed_src
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    if isinstance('tags.data', list):
        # Normalize the tags data for the current row
        tags = pd.json_normalize(tags_data)
        
        if  tags['name'].astype(str).str.contains(r'\bI&W\b', case=False, na=False).any():
            api_tags = tags[tags['name'].str.contains('API', case=False, na=False)].copy()

            if not api_tags.empty:
                for col in ['summary', 'observations', 'desriptions', 'type', 'dateAdded', 'lastModified', 'lastObserved']:
                    api_tags[col] = row.get(col)
                    
                    # Append the filtered tags to the result DataFrame
                    filtered_tags = pd.concat([filtered_tags, api_tags], ignore_index=True)
        

# Define cutoff time in UTC (timezone-aware)
cutoff = pd.Timestamp.utcnow()

# Filter to last 48 hours
recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - pd.Timedelta(hours=48)].copy()


# Extract partner names
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

# Group partners per summary
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))])
    .reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)

# Sort by most recent use
recent_tags['partners'] = partner_groups['partners']
recent_tags['partner_count'] = partner_groups['partner_count']
recent_tags = recent_tags[recent_tags['partner_count'] >= 2]
recent_tags = recent_tags[recent_tags['observations'] < 15000]
recent_tags['dateAdded'] = pd.to_datetime(recent_tags['dateAdded'])
date_Added_cutoff = pd.Timestamp.now(tz = 'UTC') - pd.Timedelta(days=30)
#recent_tags = recent_tags[recent_tags['dateAdded'] >= date_Added_cutoff]
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')
recent_tags.drop(columns=['techniqueId','tactics.data','tactics.count', 'platforms.data', 'platforms.count', 'partner', 'description' ], inplace=True)
recent_tags

KeyError: 'lastObserved'

In [198]:
tags[tags['name'].str.contains('API', case=False, na=False)]

,id,name,lastUsed,description,summary,observations,type,dateAdded,lastModified
1,35760,OS Splunk API,2025-05-06T11:27:56Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
3,35057,FDA Splunk API,2025-05-06T11:27:56Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
5,30479,CMS Splunk API,2025-05-06T11:31:10Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
7,23577,HHS Splunk API,2025-05-06T11:28:59Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z


In [199]:
filtered_tags

,id,name,lastUsed,description,techniqueId,tactics.data,tactics.count,platforms.data,platforms.count,summary,observations,type,dateAdded,lastModified
0,471298,DHA Splunk API,2025-05-06 11:30:40+00:00,CMS detected scanning activity from 45.142[.]1...,NaN,NaN,NaN,NaN,NaN,45.142.193.247,2147483647,Address,2025-03-14T11:45:58Z,2025-05-06T11:36:05Z
1,35760,OS Splunk API,2025-05-06 11:27:56+00:00,CMS detected scanning activity from 45.142[.]1...,NaN,NaN,NaN,NaN,NaN,45.142.193.247,2147483647,Address,2025-03-14T11:45:58Z,2025-05-06T11:36:05Z
2,35057,FDA Splunk API,2025-05-06 11:27:56+00:00,CMS detected scanning activity from 45.142[.]1...,NaN,NaN,NaN,NaN,NaN,45.142.193.247,2147483647,Address,2025-03-14T11:45:58Z,2025-05-06T11:36:05Z
3,30479,CMS Splunk API,2025-05-06 11:31:10+00:00,CMS detected scanning activity from 45.142[.]1...,NaN,NaN,NaN,NaN,NaN,45.142.193.247,2147483647,Address,2025-03-14T11:45:58Z,2025-05-06T11:36:05Z
4,23667,HRSA Splunk API,2025-05-06 11:28:59+00:00,CMS detected scanning activity from 45.142[.]1...,NaN,NaN,NaN,NaN,NaN,45.142.193.247,2147483647,Address,2025-03-14T11:45:58Z,2025-05-06T11:36:05Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4349,23577,HHS Splunk API,2025-05-06 11:28:59+00:00,***The following list of indicators were provi...,NaN,NaN,NaN,NaN,NaN,39.144.144.11,344,Address,2024-02-20T14:59:53Z,2025-05-02T07:35:29Z
4350,35760,OS Splunk API,2025-05-06 11:27:56+00:00,Sophos has recently reported more indicators o...,NaN,NaN,NaN,NaN,NaN,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
4351,35057,FDA Splunk API,2025-05-06 11:27:56+00:00,Sophos has recently reported more indicators o...,NaN,NaN,NaN,NaN,NaN,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
4352,30479,CMS Splunk API,2025-05-06 11:31:10+00:00,Sophos has recently reported more indicators o...,NaN,NaN,NaN,NaN,NaN,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z


In [ ]:
recent_tags[recent_tags['summary'] == 'b.pdf-fast.com']

,id,name,lastUsed,description,summary,observations,type,dateAdded,lastModified,partner_count,partners
238,471298,DHA Splunk API,2025-05-06 11:30:40+00:00,"On 4/25/2025 - MDE detected malicious file ""up...",b.pdf-fast.com,12845,Host,2025-04-28 15:15:47+00:00,2025-05-06T11:09:56Z,2,"DHA, NIH"


In [156]:
filtered_tags[filtered_tags['summary'] == '104.192.3.74']

,id,name,lastUsed,description,techniqueId,tactics.data,tactics.count,platforms.data,platforms.count,summary,observations,type,dateAdded,lastModified
50,30479,CMS Splunk API,2025-05-06 11:31:10+00:00,CMS reviewed multiple failed login attempts fr...,NaN,NaN,NaN,NaN,NaN,104.192.3.74,3947,Address,2025-04-25T13:02:39Z,2025-05-06T11:34:58Z
51,23586,IHS Splunk API,2025-05-06 11:28:59+00:00,CMS reviewed multiple failed login attempts fr...,NaN,NaN,NaN,NaN,NaN,104.192.3.74,3947,Address,2025-04-25T13:02:39Z,2025-05-06T11:34:58Z


In [195]:
tags

,id,name,lastUsed,description,summary,observations,type,dateAdded,lastModified
0,35951,Akira,2025-04-07T17:01:05Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
1,35760,OS Splunk API,2025-05-06T11:27:56Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
2,35752,VA OIS CSOC CTS Splunk,2025-05-06T11:30:40Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
3,35057,FDA Splunk API,2025-05-06T11:27:56Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
4,30770,I&W,2025-05-06T11:09:43Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
5,30479,CMS Splunk API,2025-05-06T11:31:10Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
6,23769,VA CSOC CTS Splunk,2025-05-06T06:31:02Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
7,23577,HHS Splunk API,2025-05-06T11:28:59Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
8,23576,Observed,2025-05-06T11:36:21Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z
9,553,ransomware,2025-05-06T00:47:27Z,Sophos has recently reported more indicators o...,45.227.254.26,859,Address,2024-01-29T16:31:27Z,2025-05-02T07:35:29Z


In [131]:
specific_indicator = recent_tags[recent_tags['summary'] == 'b.pdf-fast.com']
specific_indicator

,id,name,lastUsed,description,techniqueId,tactics.data,tactics.count,platforms.data,platforms.count,summary,observations,type,dateAdded,partner,partner_count
482,471298,DHA Splunk API,2025-05-06 11:30:40+00:00,"On 4/25/2025 - MDE detected malicious file ""up...",NaN,NaN,NaN,NaN,NaN,b.pdf-fast.com,12845,Host,2025-04-28T15:15:47Z,DHA,2.0
483,23630,NIH Splunk API,2025-05-06 11:30:56+00:00,"On 4/25/2025 - MDE detected malicious file ""up...",NaN,NaN,NaN,NaN,NaN,b.pdf-fast.com,12845,Host,2025-04-28T15:15:47Z,NIH,2.0


In [98]:
import os
import json
import time
import urllib.parse
import urllib3
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords

#VT_API_KEY - a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08
#OTX_API_KEY - ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe
# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Ensure NLTK stopwords are downloaded
nltk.download('stopwords')
STOP_WORDS = set(stopwords.words('english'))

# === CONFIG ===
FILE_PATH = r'C:\Users\jaskew\Documents\project_repository\data\processed\BingSearchData.json'
VT_API_KEY = "a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08"
OTX_API_KEY = "ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe"


# === Helper Functions ===

def load_saved_links(file_path):
    if not os.path.exists(file_path):
        return set()
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return {entry['link'] for entry in data if 'link' in entry}
    except (json.JSONDecodeError, IOError):
        return set()

def fetch_virustotal_data(ip):
    url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"
    headers = {"x-apikey": VT_API_KEY}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error querying VirusTotal API for {ip}: {e}")
        return None

def fetch_otx_data(ip):
    url = f"https://otx.alienvault.io/api/v1/indicators/IPv4/{ip}/general"
    headers = {"X-OTX-API-KEY": OTX_API_KEY}
    
    try:
        # First try with the correct domain
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.SSLError as ssl_err:
        print(f"SSL Error occurred: {ssl_err}")
        # If SSL verification fails, we can try with verify=False (for testing only)
        print("Attempting request with SSL verification disabled (not recommended for production)...")
        response = requests.get(url, headers=headers, verify=False)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error querying OTX API for {ip}: {e}")
        return None

def save_to_json(entry, file_path):
    try:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    data = json.load(f)
                except json.JSONDecodeError:
                    data = []
        else:
            data = []

        data.append(entry)
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
    except IOError as e:
        print(f"Error saving data: {e}")

# === Main Script ===

def main():
    saved_links = load_saved_links(FILE_PATH)
    print(f"Loaded {len(saved_links)} previously saved links.")

    search_terms = ['65.21.203.39']
    
    for indicator in search_terms:
        print(f"\n Searching for: {indicator}")
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

        # === VIRUSTOTAL ===
        vt_url = f"https://www.virustotal.com/gui/ip-address/{indicator}"
        vt_data = fetch_virustotal_data(indicator)

        if vt_data:
            attributes = vt_data.get("data", {}).get("attributes", {})
            services = attributes.get("services", [])
            ports = [s.get("port") for s in services if "port" in s]
            vt_entry = {
                'search_term': indicator,
                'timestamp': timestamp,
                'title': f"VirusTotal API Report for {indicator}",
                'link': vt_url,
                'asn': attributes.get('asn'),
                'as_owner': attributes.get('as_owner'),
                'country': attributes.get('country'),
                'network': attributes.get('network'),
                'last_analysis_stats': attributes.get('last_analysis_stats'),
                'reputation': attributes.get('reputation'),
                'total_votes': attributes.get('total_votes'),
                'open_ports': ports,
                'publish_date': "N/A"
            }

            if vt_url not in saved_links:
                save_to_json(vt_entry, FILE_PATH)
                saved_links.add(vt_url)
                print(f"Saved VT report for {indicator}.")
            else:
                print(f"VirusTotal entry for {indicator} already saved.")

        # === OTX ===
        otx_url = f"https://otx.alienvault.com/indicator/ip/{indicator}"
        otx_data = fetch_otx_data(indicator)
    
        if otx_data:
            otx_entry = {
                'search_term': indicator,
                'timestamp': timestamp,
                'title': f"OTX API Report for {indicator}",
                'link': otx_url,
                'base_indicator': otx_data.get('base_indicator', {}),
                'reputation': otx_data.get('reputation'),
               #'pulse_info': otx_data.get('pulse_info', {}),
                'geo': otx_data.get('geo', {}),
                'whois': otx_data.get('whois', {}),
                'open_ports': otx_data.get('open_ports', []),
                'publish_date': "N/A"
            }

            if otx_url not in saved_links:
                save_to_json(otx_entry, FILE_PATH)
                saved_links.add(otx_url)
                print(f"Saved OTX report for {indicator}.")
            else:
                print(f"OTX entry for {indicator} already saved.")
                

if __name__ == "__main__":
    main()


[nltk_data] Error loading stopwords: HTTP Error 503: Service
[nltk_data]     Unavailable


Loaded 0 previously saved links.

 Searching for: 65.21.203.39
Saved VT report for 65.21.203.39.
SSL Error occurred: HTTPSConnectionPool(host='otx.alienvault.io', port=443): Max retries exceeded with url: /api/v1/indicators/IPv4/65.21.203.39/general (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self signed certificate in certificate chain (_ssl.c:1002)')))
Attempting request with SSL verification disabled (not recommended for production)...
Saved OTX report for 65.21.203.39.


In [ ]:
from docx import Document

def fill_word_template(template_path, output_path, replacements):
    doc = Document(template_path)

    # Replace placeholders in paragraphs
    for para in doc.paragraphs:
        for key, value in replacements.items():
            if key in para.text:
                para.text = para.text.replace(key, value)

    # Replace placeholders in tables (if needed)
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for key, value in replacements.items():
                    if key in cell.text:
                        cell.text = cell.text.replace(key, value)

    doc.save(output_path)

# Example usage
fill_word_template(
    template_path=r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\I&W Report Template.docx",
    output_path=r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\output.docx"
,
    replacements={
        "{{title}}": "I&W Report Generated",
        "{{ip_address}}": "65.21.203.39",
        "{{indicator_types}}": "IPv4 Address",
        "{{date_observed}}": "08/01/2024",
        "{{observed_by}}": "FDA, HHS, CMS, HRSA, OS, VA",
        "{{sources}}": 'www.testResource.com',
        "{{asn_number}}": '2923',
        "{{as_owner}}": 'Hetzner Online GmbH'

    }
)
